# Milestone 3: Retrieval-Augmented Generation (RAG)

Course: Introduction to DL and GenAI Project
Competition: Smart MCQ Solver Challenge
Roll No: 22f3000114

## What this notebook does

The idea of RAG is simple. Instead of asking the model to answer from memory, we first look up a relevant fact from a knowledge base and pass it to the model as context. If the fact is useful the answer improves. If we retrieve the wrong fact the answer gets worse.

This notebook builds that pipeline one step at a time and answers the eight Milestone 3 questions along the way. The steps are:

1. Build a knowledge base from the correct answers in train.csv and index it with FAISS.
2. Retrieve the closest facts for a question using a bi-encoder.
3. Re-rank those facts with a cross-encoder to get a better order.
4. Add the best fact as context and let a zero-shot classifier pick the answer.
5. Check how often retrieval finds the right fact, and what happens when it does not.

Each question cell prints its answer after `>>> ANSWER Qn`. Run all cells on Kaggle GPU, then copy the answers into the form. The last cell prints all eight together.


## Install and imports


In [ ]:
!pip install -qU faiss-cpu sentence-transformers transformers wandb


In [ ]:
import glob, numpy as np, pandas as pd, torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

DEVICE = 0 if torch.cuda.is_available() else -1
print('CUDA available:', torch.cuda.is_available(), '| pipeline device:', DEVICE)
OPTIONS = ['A', 'B', 'C', 'D', 'E']


## Load the data


In [ ]:
def find_train():
    for pat in ['/kaggle/input/**/train.csv', './train.csv', 'train.csv']:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError('train.csv not found')

path = find_train()
print('Loading:', path)
train = pd.read_csv(path)
print('Shape:', train.shape)
train.head(2)


## Setup: build the knowledge base and FAISS index

This is the setup block from the milestone. Each document in the knowledge base is the correct option text of a row, so `kb[150]` is the true answer for row 150. We embed every document with all-MiniLM-L6-v2 and store them in a FAISS index.


In [ ]:
# Knowledge base: the correct-answer text of every row
print('Creating knowledge base')
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

# Embed the knowledge base and build the FAISS index
print('Loading embedding model and creating index')
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda' if DEVICE == 0 else 'cpu')
kb_embeddings = model.encode(kb, show_progress_bar=False)
kb_embeddings = np.asarray(kb_embeddings, dtype='float32')
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print('Knowledge base created:', index.ntotal, 'vectors, dim', kb_embeddings.shape[1])


## Helpers

`map_at_3` is the same metric used in Milestone 1 and 2, with its assertion tests. `retrieve` embeds a prompt and returns the top-k documents from FAISS.


In [ ]:
def map_at_3(predictions, labels):
    weights = [1.0, 1/2, 1/3]
    total = 0.0
    for preds, true in zip(predictions, labels):
        s = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                s = weights[i]; break
        total += s
    return total / len(labels)

assert abs(map_at_3([['A','B','C']], ['A']) - 1.0)  < 1e-9
assert abs(map_at_3([['A','B','C']], ['B']) - 0.5)  < 1e-9
assert abs(map_at_3([['A','B','C']], ['C']) - 1/3)  < 1e-9
assert abs(map_at_3([['A','B','C']], ['D']) - 0.0)  < 1e-9
print('map_at_3 tests passed.')

def retrieve(prompt, k):
    q = np.asarray(model.encode([str(prompt)]), dtype='float32')
    D, I = index.search(q, k)
    return I[0].tolist(), D[0].tolist()


## Zero-shot classifier and the row 150 setup

facebook/bart-large-mnli is the zero-shot model from the bootcamp. The candidate labels for a question are the five option texts, and the classifier returns a probability over them. We often need the score it gives to the correct option, so there is a small helper for that.


In [ ]:
zs = pipeline('zero-shot-classification', model='facebook/bart-large-mnli', device=DEVICE)

row_150    = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150[o]) for o in OPTIONS]      # the 5 option texts
ans_150    = str(row_150[row_150['answer']])         # text of the correct option

def correct_option_score(zs_result, correct_text):
    return dict(zip(zs_result['labels'], zs_result['scores']))[correct_text]


## Q1: Zero-shot score with no context

We give the model the prompt for row 150 and the five options as candidate labels, then read the probability it gives to the correct option. There is no context yet, so this is the model answering from its own knowledge.


In [ ]:
res_q1 = zs(prompt_150, labels_150)
q1 = round(correct_option_score(res_q1, ans_150), 3)
print('>>> ANSWER Q1 (prob of correct option, no context):', q1)


## Q2: Where FAISS ranks the true document

We embed the row 150 prompt and ask FAISS for the 10 closest documents, then check the position of the true document (index 150). A bi-encoder embeds the question and the documents separately, so the true document is often not first.


In [ ]:
idx_q2, dist_q2 = retrieve(prompt_150, k=10)
q2 = idx_q2.index(150) + 1 if 150 in idx_q2 else 'not in top 10'
print('Retrieved indices:', idx_q2)
print('>>> ANSWER Q2 (rank of true doc in FAISS top-10):', q2)


## Q3: Re-ranking with a cross-encoder

A cross-encoder reads the question and a document together in one pass, so it judges relevance more accurately than the bi-encoder. We take the 10 documents from Q2, score each one against the prompt, sort by that score, and check the new rank of the true document.


In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in idx_q2]
pairs   = [[prompt_150, d] for d in docs_10]
ce_scores = cross_encoder.predict(pairs)

order = np.argsort(-np.asarray(ce_scores))       # positions into idx_q2, best first
reranked_indices = [idx_q2[p] for p in order]
q3 = reranked_indices.index(150) + 1 if 150 in reranked_indices else 'not present'
print('Re-ranked KB indices:', reranked_indices)
print('>>> ANSWER Q3 (rank of true doc after cross-encoder):', q3)


## Q4: How long the RAG prompt gets

We retrieve the top 5 documents for row 42, join them with spaces, and build the string `"Context: ... Question: ..."`. Then we count how many tokens the bert-base-uncased tokenizer produces. This shows how fast retrieved context fills up the model's input.

The form asks for the token count. `tokenizer.tokenize` gives the word-piece count on its own. Encoding the same string adds the `[CLS]` and `[SEP]` tokens, so it is two more. Both are printed. Try the plain word-piece count first.


In [ ]:
prompt_42 = str(train.iloc[42]['prompt'])
idx_42, _ = retrieve(prompt_42, k=5)
concat_docs = ' '.join(kb[i] for i in idx_42)
rag_string_42 = f'Context: {concat_docs} Question: {prompt_42}'

bert_tok = AutoTokenizer.from_pretrained('bert-base-uncased')
n_wordpieces   = len(bert_tok.tokenize(rag_string_42))
n_with_special = len(bert_tok(rag_string_42, truncation=False)['input_ids'])
print('word-piece tokens (tokenizer.tokenize):', n_wordpieces)
print('encoded length with [CLS] and [SEP]  :', n_with_special)
print('>>> ANSWER Q4 (total tokens):', n_wordpieces, '(with special tokens:', n_with_special, ')')


## Q5: Zero-shot score with the correct context

Now we pass the true document for row 150 as context and run the same zero-shot classification. Compare this probability to Q1 to see whether the right context helped.


In [ ]:
true_doc_150 = kb[150]
rag_good = f'Context: {true_doc_150} Question: {prompt_150}'
res_q5 = zs(rag_good, labels_150)
q5 = round(correct_option_score(res_q5, ans_150), 3)
print('>>> ANSWER Q5 (prob of correct option, correct context):', q5)
print('    no context Q1 =', q1, ' correct context Q5 =', q5)


## Q6: What happens with the wrong context

Here we deliberately feed an unrelated fact (document 999) as context. The model tends to trust whatever context we give it, so a wrong fact can pull the probability of the correct option down. This is why retrieval quality matters.


In [ ]:
adv_doc = kb[999]
rag_bad = f'Context: {adv_doc} Question: {prompt_150}'
res_q6 = zs(rag_bad, labels_150)
q6 = round(correct_option_score(res_q6, ans_150), 3)
print('>>> ANSWER Q6 (prob of correct option, wrong context):', q6)
print('    no context:', q1, ' correct context:', q5, ' wrong context:', q6)


## Q7: How often retrieval finds the answer

For the first 100 rows we retrieve the top 5 documents and check whether the row's correct answer text appears in any of them. The hit rate tells us the ceiling for RAG. If the fact is never retrieved, no reader can use it.


In [ ]:
hits = 0
for i in range(100):
    correct_str = str(train.iloc[i][train.iloc[i]['answer']])
    ret_idx, _ = retrieve(str(train.iloc[i]['prompt']), k=5)
    ret_docs = [kb[j] for j in ret_idx]
    if any(correct_str in d for d in ret_docs):
        hits += 1
q7 = round(hits / 100 * 100, 1)
print('>>> ANSWER Q7 (hit rate percent, first 100 rows):', q7)


## Q8: The full RAG pipeline scored with MAP@3

For the first 20 rows we run the whole pipeline: retrieve 5, re-rank and keep the best one, add it as context, run zero-shot over the 5 options, rank the letters, and score MAP@3. This is the complete two-stage RAG setup on the competition metric.


In [ ]:
predictions, truths = [], []
for i in range(20):
    row    = train.iloc[i]
    prompt = str(row['prompt'])
    opt_texts = [str(row[o]) for o in OPTIONS]
    letter_of = {str(row[o]): o for o in OPTIONS}   # option text back to its letter

    ret_idx, _ = retrieve(prompt, k=5)              # retrieve
    cand_docs  = [kb[j] for j in ret_idx]
    ce = cross_encoder.predict([[prompt, d] for d in cand_docs])   # re-rank
    best_doc = cand_docs[int(np.argmax(ce))]
    rag = f'Context: {best_doc} Question: {prompt}'                # augment
    r = zs(rag, opt_texts)                                         # predict
    ranked_letters = [letter_of[t] for t in r['labels']]          # already sorted by score
    predictions.append(ranked_letters[:3])
    truths.append(row['answer'])

q8 = round(map_at_3(predictions, truths), 3)
print('>>> ANSWER Q8 (RAG pipeline MAP@3, 20 rows):', q8)


## Log the run to Weights and Biases

We log this to the same project as Milestone 1 and 2 so all the runs sit together for the viva. We record the pipeline MAP@3 and the retrieval hit rate.


In [ ]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB'))
    WANDB_OK = True
except Exception as e:
    print('W&B secret not found, skipping logging.', e)
    WANDB_OK = False

if WANDB_OK:
    run = wandb.init(entity='22f3000114-dl-genai-project', project='dl-genai-project',
                     name='m3-rag-two-stage', group='milestone-3',
                     config={'milestone': 3, 'retriever': 'all-MiniLM-L6-v2',
                             'reranker': 'ms-marco-MiniLM-L-6-v2',
                             'reader': 'facebook/bart-large-mnli', 'k': 5}, reinit=True)
    run.log({'map_at_3': q8, 'retrieval_hit_rate': q7,
             'row150_prob_nocontext': q1, 'row150_prob_goodcontext': q5,
             'row150_prob_advcontext': q6})
    run.finish()
    print('Logged m3-rag-two-stage to W&B.')


## Notes on the results

Fill this in after running.

The A/B test is the main point of the milestone. Look at the correct-option probability for row 150 in three cases: no context (Q1), correct context (Q5), and wrong context (Q6). Good context should raise it, wrong context should lower it.

Q2 and Q3 show why we re-rank. FAISS gives a fast but rough order, and the cross-encoder cleans it up.

Q7 is the retrieval ceiling and Q8 is the result of the full pipeline. Compare Q8 to the Milestone 1 majority-class score of 0.4213 to see if the RAG pipeline does better.


## All form answers together


In [ ]:
print('Milestone 3 form answers')
print('  Q1 (prob correct, no context)        :', q1)
print('  Q2 (FAISS rank of true doc)          :', q2)
print('  Q3 (cross-encoder rank of true doc)  :', q3)
print('  Q4 (tokens, row 42 RAG string)       :', n_wordpieces, '(with special tokens:', n_with_special, ')')
print('  Q5 (prob correct, correct context)   :', q5)
print('  Q6 (prob correct, wrong context)     :', q6)
print('  Q7 (hit rate percent, 100 rows)      :', q7)
print('  Q8 (RAG pipeline MAP@3, 20 rows)     :', q8)
